In [1]:
trash_classes = {

    "trash branch":0,
    "trash etc":1,
    "trash fabric":2,
    "trash fishing gear":3,
    "trash metal":4,
    "trash net":5,
    "trash paper":6,
    "trash plastic":7,
    "trash rope":8,
    "trash rubber":9,
    "trash tarp":10,
    "trash unknown instance":11,
    "trash wood":12,
    "trash wreckage":13
}

In [2]:
import os

folders = [

    "YOLO_Dataset/images/train",
    "YOLO_Dataset/images/val",

    "YOLO_Dataset/labels/train",
    "YOLO_Dataset/labels/val"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("YOLO Dataset Folders Created")

YOLO Dataset Folders Created


In [3]:
import os

ann_folder = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance train\ann"

files = os.listdir(ann_folder)

print(files[:10])

['vid_000002_frame0000023.jpg.json', 'vid_000003_frame0000007.jpg.json', 'vid_000003_frame0000008.jpg.json', 'vid_000003_frame0000010.jpg.json', 'vid_000003_frame0000011.jpg.json', 'vid_000003_frame0000012.jpg.json', 'vid_000003_frame0000013.jpg.json', 'vid_000003_frame0000014.jpg.json', 'vid_000003_frame0000015.jpg.json', 'vid_000003_frame0000016.jpg.json']


In [4]:
img_folder = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance train\img"

imgs = os.listdir(img_folder)

print(imgs[:10])

['vid_000002_frame0000023.jpg', 'vid_000003_frame0000007.jpg', 'vid_000003_frame0000008.jpg', 'vid_000003_frame0000010.jpg', 'vid_000003_frame0000011.jpg', 'vid_000003_frame0000012.jpg', 'vid_000003_frame0000013.jpg', 'vid_000003_frame0000014.jpg', 'vid_000003_frame0000015.jpg', 'vid_000003_frame0000016.jpg']


In [5]:
import os
import json
import shutil
from tqdm import tqdm

In [6]:
trash_classes = {

    "trash branch":0,
    "trash etc":1,
    "trash fabric":2,
    "trash fishing gear":3,
    "trash metal":4,
    "trash net":5,
    "trash paper":6,
    "trash plastic":7,
    "trash rope":8,
    "trash rubber":9,
    "trash tarp":10,
    "trash unknown instance":11,
    "trash wood":12,
    "trash wreckage":13
}

In [7]:
folders = [

    "YOLO_Dataset/images/train",
    "YOLO_Dataset/images/val",

    "YOLO_Dataset/labels/train",
    "YOLO_Dataset/labels/val"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("YOLO Dataset Structure Created")

YOLO Dataset Structure Created


In [8]:
def convert_annotation(
    ann_file,
    image_file,
    label_output
):

    with open(ann_file, "r") as f:
        data = json.load(f)

    img_w = data["size"]["width"]
    img_h = data["size"]["height"]

    yolo_lines = []

    for obj in data["objects"]:

        # ----------------------------------
        # Keep only trash classes
        # ----------------------------------

        cls_name = obj["classTitle"]

        if cls_name not in trash_classes:
            continue

        # ----------------------------------
        # Keep rectangle annotations only
        # ----------------------------------

        if obj["geometryType"] != "rectangle":
            continue

        cls_id = trash_classes[cls_name]

        points = obj["points"]["exterior"]

        x1 = points[0][0]
        y1 = points[0][1]

        x2 = points[1][0]
        y2 = points[1][1]

        # ----------------------------------
        # Convert to YOLO
        # ----------------------------------

        x_center = ((x1+x2)/2)/img_w
        y_center = ((y1+y2)/2)/img_h

        width = abs(x2-x1)/img_w
        height = abs(y2-y1)/img_h

        yolo_lines.append(
            f"{cls_id} {x_center} {y_center} {width} {height}"
        )

    with open(label_output, "w") as f:

        for line in yolo_lines:
            f.write(line + "\n")

In [9]:
instance_train_img = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance train\img"

instance_train_ann = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance train\ann"

files = os.listdir(instance_train_ann)

count = 0

for ann_file in tqdm(files):

    ann_path = os.path.join(
        instance_train_ann,
        ann_file
    )

    img_name = ann_file.replace(".json","")

    img_path = os.path.join(
        instance_train_img,
        img_name
    )

    if not os.path.exists(img_path):
        continue

    label_path = os.path.join(
        "YOLO_Dataset/labels/train",
        img_name.replace(".jpg",".txt")
    )

    convert_annotation(
        ann_path,
        img_path,
        label_path
    )

    shutil.copy(
        img_path,
        "YOLO_Dataset/images/train"
    )

    count += 1

print("Train Images Processed:", count)

100%|██████████████████████████████████████████████████████████████████████████████| 6065/6065 [02:05<00:00, 48.47it/s]

Train Images Processed: 6065


In [10]:
instance_val_img = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance val\img"

instance_val_ann = r"E:\PHD Rahul Jain\Underwater\trashcan_data\instance val\ann"

files = os.listdir(instance_val_ann)

count = 0

for ann_file in tqdm(files):

    ann_path = os.path.join(
        instance_val_ann,
        ann_file
    )

    img_name = ann_file.replace(".json","")

    img_path = os.path.join(
        instance_val_img,
        img_name
    )

    if not os.path.exists(img_path):
        continue

    label_path = os.path.join(
        "YOLO_Dataset/labels/val",
        img_name.replace(".jpg",".txt")
    )

    convert_annotation(
        ann_path,
        img_path,
        label_path
    )

    shutil.copy(
        img_path,
        "YOLO_Dataset/images/val"
    )

    count += 1

print("Validation Images Processed:", count)

100%|██████████████████████████████████████████████████████████████████████████████| 1147/1147 [00:17<00:00, 64.83it/s]

Validation Images Processed: 1147


In [11]:
yaml_content = """
path: YOLO_Dataset

train: images/train
val: images/val

names:

  0: trash branch
  1: trash etc
  2: trash fabric
  3: trash fishing gear
  4: trash metal
  5: trash net
  6: trash paper
  7: trash plastic
  8: trash rope
  9: trash rubber
  10: trash tarp
  11: trash unknown instance
  12: trash wood
  13: trash wreckage
"""

with open(
    "YOLO_Dataset/data.yaml",
    "w"
) as f:
    f.write(yaml_content)

print("data.yaml Created")

data.yaml Created


In [12]:
train_imgs = len(
    os.listdir(
        "YOLO_Dataset/images/train"
    )
)

val_imgs = len(
    os.listdir(
        "YOLO_Dataset/images/val"
    )
)

train_labels = len(
    os.listdir(
        "YOLO_Dataset/labels/train"
    )
)

val_labels = len(
    os.listdir(
        "YOLO_Dataset/labels/val"
    )
)

print("Train Images :", train_imgs)
print("Train Labels :", train_labels)

print("Val Images :", val_imgs)
print("Val Labels :", val_labels)

Train Images : 6065
Train Labels : 6065
Val Images : 1147
Val Labels : 1147
